In [1]:
!nvidia-smi
import torch

print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(
        f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB"
    )

Tue Jun  9 19:11:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 1: Clone Repository and Setup Environment

In [ ]:
import os
from pathlib import Path

# Configuration
REPO_URL = "https://github.com/sattary/ali_proj.git"
BRANCH = "alis_code"  # Change if using different branch
PROJECT_DIR = "ali_proj"

# 1. Clone Repository
if not Path(PROJECT_DIR).exists():
    print(f"Cloning {REPO_URL} (branch: {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL}
else:
    print("Repository already cloned. Pulling latest changes...")
    !cd {PROJECT_DIR} && git pull origin {BRANCH}

%cd {PROJECT_DIR}

# 2. Install uv
print("\nInstalling uv...")
!pip install -q uv

# 3. Set MPLBACKEND for Kaggle compatibility
os.environ['MPLBACKEND'] = 'Agg'
print("\nSet MPLBACKEND=Agg for Kaggle compatibility")

# 4. Sync Dependencies
print("\nSyncing dependencies...")
!uv sync

print("\n✓ Setup complete!")

In [ ]:
# Verify Git Repository
from pathlib import Path
import subprocess

# Check if we're in a git repo
result = subprocess.run(
    ["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True
)
if result.returncode == 0:
    repo_root = result.stdout.strip()
    print(f"✓ Git repository found at: {repo_root}")
    print(f"✓ Current directory: {Path.cwd()}")
else:
    print("Error: Not in a git repository!")
    print(f"Current directory: {Path.cwd()}")
    raise RuntimeError("Git repository not found")

# Show repo status
!git status

## Phase 1: Deterministic Generation (`generate`)

Generate the synthetic interferogram dataset to HDF5 shards. The cryptographic seed guarantees mathematically invariant noise topologies.

In [ ]:
# Phase 1: Generate Data
!uv run phase-unwrap generate \
    --num-samples 180000 \
    --shard-size 1000 \
    --out-dir data/kaggle_full \
    --seed 1337

print("\n✓ Data generation complete!")

## Phase 2: Hyperparameter Optimization (`tune`)

Use Optuna's Bayesian TPE algorithm to isolate the absolute lowest-error configuration. The best configuration is automatically saved to `runs/optuna/best_config.yaml`.

In [ ]:
# Phase 2: Optuna Tune
# Automatically parallelizes across available GPUs
!uv run phase-unwrap tune \
    --use-amp \
    --n-trials 30 \
    --tune-epochs 15 \
    --study-name kaggle_hpo \
    --n-workers 2

print("\n✓ Tuning complete!")
print("Best config frozen to: runs/optuna/best_config.yaml")

## Phase 3: Primary Training and Evaluation (`train`)

Train the baseline network to convergence using the frozen `best_config.yaml`.

In [ ]:
# Phase 3: Train Primary Network
import os
config_arg = "--config runs/optuna/best_config.yaml" if os.path.exists("runs/optuna/best_config.yaml") else ""

!uv run phase-unwrap train \
    --use-amp \
    {config_arg} \
    --run-name exp_primary \
    --multi-gpu

print("\n✓ Primary training complete!")

## Phase 4: Statistical Validation (`multiseed`)

Defend against 'lucky seed' anomalies. Spawns completely independent training convergences using the same frozen configuration, and automatically aggregates the metrics into Mean ± Std.

In [ ]:
# Phase 4: Multi-Seed Aggregation
!uv run phase-unwrap multiseed \
    --use-amp \
    {config_arg} \
    --run-name exp_multiseed \
    --num-seeds 3 \
    --multi-gpu

print("\n✓ Multi-seed validation complete!")

## Phase 5: Architectural Ablation (`ablation`)

Mathematically prove the necessity of your custom topology by systematically crippling the network. Exports a rigorous LaTeX comparison table.

In [ ]:
# Phase 5: Ablation Sweeps
!uv run phase-unwrap ablation \
    --use-amp \
    {config_arg} \
    --run-name exp_ablation \
    --num-seeds 3 \
    --multi-gpu \
    --out-table results/tables/ablation.tex

print("\n✓ Ablation sweeps complete!")

## Step 6: Zip and Download Results

Run this cell to zip the `runs/` and `results/` directories so you can render them on your local machine.

In [ ]:
import shutil
from IPython.display import FileLink

print("Zipping runs and results...")
shutil.make_archive('training_results', 'zip', 'runs/')
shutil.make_archive('tables_results', 'zip', 'results/')
print("✓ Done!")
display(FileLink('training_results.zip'))
display(FileLink('tables_results.zip'))